# Lab Assignment 1: Setting Up Python NLP Tools and Building a Basic Text Processing Pipeline

**Course:** AAI 6620 — Applied Natural Language Processing

**Student:** Muhammad Obaidullah (abdulsalam.m@northeastern.edu)

**Submission:** Upload `Lab_1.ipynb` to Gradescope (via Canvas) with all cells run and outputs visible.

---

## Overview

In this lab you will set up the core Python libraries used throughout this course and build a basic text analysis pipeline from scratch. You will work with NumPy for numerical operations on text feature data, pandas for loading and manipulating a text dataset, and NLTK for computing frequency statistics over a corpus.

The dataset is a small collection of product reviews. This domain is intentionally simple so that you can focus on the mechanics of the tools rather than the subject matter. These same tools and patterns will appear in every module of the course.

---

## Learning Objectives

By the end of this lab you will be able to:

1. Set up a working Python NLP environment using NumPy, pandas, and NLTK
2. Use NumPy for efficient numerical operations on text feature data
3. Load, inspect, and manipulate a text dataset using pandas
4. Compute a frequency distribution for an NLP corpus using NLTK

---

## How this notebook is organised

Parts 1–3 are the graded scaffold, completed exactly as specified — the named
variables the autograder checks are produced with the names given, and the three
`do not change` constants and the dataset cell are untouched.

Parts 4 and 5 cover the four things the Canvas brief asks for that the scaffold
has no stub for: a missing-value audit and cleaning step, **at least two** derived
columns, the **top 20** words rather than the top 10, and the whole thing assembled
into one documented pipeline.

Part 4 works on `reviews_df`, a copy of `df`, on purpose: `df` is left at exactly
the shape the graded tasks produced it in, so adding my own feature columns cannot
change an answer that has already been checked.

---

## Grading

| Component | Points | Method |
|---|---|---|
| Part 1 — NumPy operations | 22 | Autograded |
| Part 2 — pandas operations | 30 | Autograded |
| Part 3 — NLTK frequency analysis | 23 | Autograded |
| Code quality | 25 | Manual review |
| **Total** | **100** | |

### Manual grading rubric (25 points)

The following criteria are assessed by your instructor. These are the professional standards expected of a working data scientist, MLOps engineer, or NLP practitioner. **A note on expectations:** the bar for manual grading rises as the course progresses. You may receive full or near-full marks on these criteria in Lab 1 even with minor gaps — but the same gaps will result in deductions in later labs. Build good habits now.

| Criterion | Points | What we look for |
|---|---|---|
| Docstrings and inline comments | 6 | Every function has a docstring describing purpose, parameters, and return value. Non-obvious logic has a brief comment. Trivial lines do not need comments. |
| Function design | 6 | Logic is wrapped in reusable functions with clear inputs and outputs. No values that should be parameters are hard-coded inside functions. Each function does one thing. |
| Defensive coding | 5 | Basic input validation or type checking where it matters. Code does not silently fail on unexpected input. This matters for pipeline stability. |
| Code clarity and naming | 4 | Variable and function names are descriptive and self-documenting. No single-letter names outside of tight loops. Code reads like a narrative, not a transcript. |
| Avoiding redundancy | 4 | No copy-pasted logic that should be a function. No unnecessary intermediate variables. Apply the DRY (Don't Repeat Yourself) principle proportionally. |

---

## Setup

Run the cell below to install and verify your environment. In Google Colab, NLTK is pre-installed but the corpora must be downloaded each session.

In [121]:
# Setup — run this cell first every session
# It installs any missing packages and downloads the NLTK data needed for this lab.

import subprocess, sys

def install_if_missing(package):
    """Install a package if it cannot be imported.

    Args:
        package (str): The package name to check and install.
    """
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

install_if_missing("numpy")
install_if_missing("pandas")
install_if_missing("nltk")

import numpy as np
import pandas as pd
import nltk

# Download required NLTK resources
for resource in ["punkt", "punkt_tab", "stopwords", "brown"]:
    nltk.download(resource, quiet=True)

print("Environment ready.")
print(f"  numpy  {np.__version__}")
print(f"  pandas {pd.__version__}")
print(f"  nltk   {nltk.__version__}")
print(f"  python {sys.version.split()[0]}")

Environment ready.
  numpy  2.2.6
  pandas 2.3.3
  nltk   3.10.3
  python 3.10.13


---

## Part 1 — NumPy: Numerical Operations on Text Feature Data (22 points)

NumPy is the numerical backbone of the Python data science stack. In NLP, you will use it constantly to work with arrays of feature values — word counts, TF-IDF scores, embeddings, and more.

In this part you will work with the `rating` and `helpful_votes` columns from the review dataset as NumPy arrays. Each task asks you to produce a specific named variable. The autograder checks those variable names, so use them exactly as specified.

### 1.1 — Load the dataset and extract arrays

Run the cell below to load the review dataset. You do not need to modify it.

In [122]:
# Dataset — do not modify this cell.
# This is a small collection of product reviews used throughout the lab.

import io

REVIEWS_CSV = """review_id,text,rating,category,helpful_votes
1,Great sound quality and comfortable fit,5,electronics,42
2,Stopped working after two weeks,1,electronics,88
3,Exactly as described fast shipping,4,books,15
4,Poor build quality feels cheap,2,home,31
5,Absolutely love this product highly recommend,5,clothing,67
6,Instructions were confusing but product works,3,home,22
7,Battery life is impressive lasts all day,5,electronics,54
8,Returned immediately not as pictured,1,clothing,93
9,Good value for the price,4,books,18
10,Fits perfectly true to size,5,clothing,29
11,Broke on first use very disappointed,1,home,71
12,Decent quality nothing special,3,electronics,8
13,Beautifully written compelling story,5,books,45
14,Too small even for the size listed,2,clothing,37
15,Works as expected no complaints,4,home,12
16,Overpriced for what you get,2,electronics,60
17,Couldnt put it down finished in one sitting,5,books,83
18,Delivery was fast packaging was damaged,3,home,26
19,Great gift idea well received,4,clothing,19
20,Sound cuts out randomly frustrating,2,electronics,48
"""

df = pd.read_csv(io.StringIO(REVIEWS_CSV))
print(f"Loaded {len(df)} reviews across {df['category'].nunique()} categories.")
df.head()

Loaded 20 reviews across 4 categories.


,review_id,text,rating,category,helpful_votes
0,1,Great sound quality and comfortable fit,5,electronics,42
1,2,Stopped working after two weeks,1,electronics,88
2,3,Exactly as described fast shipping,4,books,15
3,4,Poor build quality feels cheap,2,home,31
4,5,Absolutely love this product highly recommend,5,clothing,67


### 1.1b — Shared validation helpers

Six of the functions below need the same two checks: *is this the type I think it
is*, and *does this frame have the columns I am about to index*. Writing those
checks once here, instead of six times inline, is the DRY point from the rubric —
and it means an unexpected input produces a named error at the call site rather
than a `KeyError` from somewhere deep in pandas.

In [123]:
# Shared validation helpers, used by every task below.

def require_columns(dataframe, *column_names):
    """Assert that a value is a DataFrame holding the named columns.

    Args:
        dataframe (pd.DataFrame): The frame to validate.
        *column_names (str): Columns that must be present. May be empty, in
            which case only the type is checked.

    Returns:
        pd.DataFrame: The same object, so a caller can validate inline.

    Raises:
        TypeError: If `dataframe` is not a pandas DataFrame.
        KeyError: If any requested column is missing.
    """
    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(f"Expected a pandas DataFrame, got {type(dataframe).__name__}.")
    missing = [name for name in column_names if name not in dataframe.columns]
    if missing:
        raise KeyError(f"Missing column(s) {missing}; available: {list(dataframe.columns)}")
    return dataframe


def require_nonempty_array(array, name="array"):
    """Assert that a value is a non-empty NumPy array.

    Args:
        array (np.ndarray): The array to validate.
        name (str): Name to use in error messages, so the caller can tell which
            argument was wrong.

    Returns:
        np.ndarray: The same object.

    Raises:
        TypeError: If `array` is not an ndarray.
        ValueError: If `array` has no elements.
    """
    if not isinstance(array, np.ndarray):
        raise TypeError(f"{name} must be a numpy.ndarray, got {type(array).__name__}.")
    if array.size == 0:
        raise ValueError(f"{name} is empty; there is nothing to compute.")
    return array


print("Helpers defined: require_columns, require_nonempty_array")

Helpers defined: require_columns, require_nonempty_array


### 1.2 — Create NumPy arrays

Extract the `rating` and `helpful_votes` columns as NumPy arrays.

**Your task:** Complete the function below and call it to produce `ratings_array` and `helpful_array`.

In [124]:
def extract_arrays(dataframe, col_a, col_b):
    """Extract two DataFrame columns as NumPy arrays, in row order.

    Args:
        dataframe (pd.DataFrame): Source frame.
        col_a (str): Name of the first column to extract.
        col_b (str): Name of the second column to extract.

    Returns:
        tuple[np.ndarray, np.ndarray]: The values of `col_a` and `col_b`.

    Raises:
        TypeError: If `dataframe` is not a DataFrame.
        KeyError: If either column is absent.
    """
    require_columns(dataframe, col_a, col_b)
    # .to_numpy() is the documented accessor. .values is legacy and does not
    # promise an ndarray back for extension dtypes, which pandas 3.0 now uses
    # for string columns by default.
    return dataframe[col_a].to_numpy(), dataframe[col_b].to_numpy()

ratings_array, helpful_array = extract_arrays(df, 'rating', 'helpful_votes')

# Verify
print(f"ratings_array: {ratings_array}")
print(f"helpful_array: {helpful_array}")
print(f"dtype: {ratings_array.dtype}, shape: {ratings_array.shape}")

ratings_array: [5 1 4 2 5 3 5 1 4 5 1 3 5 2 4 2 5 3 4 2]
helpful_array: [42 88 15 31 67 22 54 93 18 29 71  8 45 37 12 60 83 26 19 48]
dtype: int64, shape: (20,)


### 1.3 — Summary statistics

Compute basic statistics on `ratings_array` using NumPy methods.

**Your task:** Complete the function and produce the named variables below.

In [125]:
def compute_array_stats(arr):
    """Summarise a numeric array with its observation count and arithmetic mean.

    "Total" is read here as the number of ratings, not the sum of their values.
    The brief does not define the term, so the reading is stated here explicitly:
    the count of observations, not their total.

    Args:
        arr (np.ndarray): Non-empty numeric array.

    Returns:
        tuple[int, np.floating]: (number of observations, mean of values).

    Raises:
        TypeError: If `arr` is not an ndarray.
        ValueError: If `arr` is empty, which would make the mean undefined.
    """
    require_nonempty_array(arr, "arr")
    # .size counts every element in the array. len() would return only the
    # length of the first axis, which is the same thing for the 1-D arrays here
    # and quietly wrong the moment the input gains a second dimension.
    return arr.size, arr.mean()

total_ratings, mean_rating = compute_array_stats(ratings_array)

print(f"total_ratings: {total_ratings}")
print(f"mean_rating:   {mean_rating}")

total_ratings: 20
mean_rating:   3.3


### 1.4 — Normalization

Normalize `ratings_array` so that all values fall in the range [0, 1] by dividing by the maximum value.

**Your task:** Complete the function and produce `normalized_ratings`.

In [126]:
def normalize_array(arr):
    """Scale an array into [0, 1] by dividing it through by its maximum.

    This is max-scaling, not min-max scaling. It preserves the ratio between any
    two values, but it only reaches 0 if the array actually contains a 0 — so on
    a 1-5 rating scale the smallest result is 0.2, not 0.0. That is the correct
    behaviour when the scale has a meaningful zero point; min-max scaling would
    throw away the fact that a 1-star review is not "nothing".

    Args:
        arr (np.ndarray): Non-empty numeric array with a non-zero maximum.

    Returns:
        np.ndarray: Float array of the same shape, divided by the maximum.

    Raises:
        ValueError: If `arr` is empty or its maximum is 0.
    """
    require_nonempty_array(arr, "arr")
    maximum = arr.max()
    if maximum == 0:
        raise ValueError("Cannot normalize by a maximum of 0 — every value is 0.")
    return arr / maximum

normalized_ratings = normalize_array(ratings_array)

print(f"normalized_ratings: {normalized_ratings}")
print(f"min: {normalized_ratings.min():.4f}, max: {normalized_ratings.max():.4f}")

normalized_ratings: [1.  0.2 0.8 0.4 1.  0.6 1.  0.2 0.8 1.  0.2 0.6 1.  0.4 0.8 0.4 1.  0.6
 0.8 0.4]
min: 0.2000, max: 1.0000


### 1.5 — Boolean masking and filtering

Identify reviews whose rating is above a given threshold and extract them.

**Your task:** Complete the two functions below. Produce `above_threshold_mask` and `filtered_ratings`.

In [127]:
RATING_THRESHOLD = 3  # do not change

def make_threshold_mask(arr, threshold):
    """Build a Boolean mask marking elements strictly above a threshold.

    "Above" is read strictly, so a value equal to `threshold` is excluded.

    Args:
        arr (np.ndarray): Non-empty numeric array.
        threshold (int | float): Value to compare against.

    Returns:
        np.ndarray: Boolean array of the same shape as `arr`.

    Raises:
        TypeError: If `threshold` is not numeric.
    """
    require_nonempty_array(arr, "arr")
    if not isinstance(threshold, (int, float, np.number)) or isinstance(threshold, bool):
        raise TypeError(f"threshold must be numeric, got {type(threshold).__name__}.")
    return arr > threshold

def apply_mask(arr, mask):
    """Select the elements of an array where a Boolean mask is True.

    Args:
        arr (np.ndarray): Array to select from.
        mask (np.ndarray): Boolean array of the same shape as `arr`.

    Returns:
        np.ndarray: A new 1-D array of the selected elements.

    Raises:
        TypeError: If `mask` is not a Boolean array.
        ValueError: If `mask` does not match the shape of `arr`.
    """
    require_nonempty_array(arr, "arr")
    if getattr(mask, "dtype", None) != np.bool_:
        raise TypeError("mask must be a Boolean ndarray; an integer array would "
                        "be treated as fancy indexing and select the wrong rows.")
    if mask.shape != arr.shape:
        raise ValueError(f"Shape mismatch: mask {mask.shape} vs arr {arr.shape}.")
    return arr[mask]

above_threshold_mask = make_threshold_mask(ratings_array, RATING_THRESHOLD)
filtered_ratings = apply_mask(ratings_array, above_threshold_mask)

print(f"Mask:             {above_threshold_mask}")
print(f"filtered_ratings: {filtered_ratings}")
print(f"Count above threshold: {filtered_ratings.size}")

Mask:             [ True False  True False  True False  True False  True  True False False
  True False  True False  True False  True False]
filtered_ratings: [5 4 5 5 4 5 5 4 5 4]
Count above threshold: 10


### 1.6 — Correlation

Compute the Pearson correlation between `ratings_array` and `helpful_array`. This tells you whether higher-rated reviews tend to receive more helpful votes (or fewer).

**Your task:** Complete the function and produce `ratings_helpful_corr`. Round to 4 decimal places.

In [128]:
def compute_correlation(arr_a, arr_b, ndigits=4):
    """Pearson correlation coefficient between two equal-length arrays.

    Args:
        arr_a (np.ndarray): First numeric array.
        arr_b (np.ndarray): Second numeric array, same shape as `arr_a`.
        ndigits (int): Decimal places to round to. Defaults to 4.

    Returns:
        float: Pearson r, rounded to `ndigits`.

    Raises:
        ValueError: If either array is empty or the shapes differ.
    """
    require_nonempty_array(arr_a, "arr_a")
    require_nonempty_array(arr_b, "arr_b")
    if arr_a.shape != arr_b.shape:
        raise ValueError(f"Shape mismatch: {arr_a.shape} vs {arr_b.shape}.")
    # np.corrcoef returns the full 2x2 correlation matrix; r is the off-diagonal.
    return round(float(np.corrcoef(arr_a, arr_b)[0, 1]), ndigits)

ratings_helpful_corr = compute_correlation(ratings_array, helpful_array)

print(f"Pearson correlation (rating vs helpful_votes): {ratings_helpful_corr}")
print("Interpretation: ", end="")
if ratings_helpful_corr > 0.1:
    print("Higher-rated reviews tend to get more helpful votes.")
elif ratings_helpful_corr < -0.1:
    print("Lower-rated reviews tend to get more helpful votes.")
else:
    print("No strong linear relationship.")

Pearson correlation (rating vs helpful_votes): -0.3168
Interpretation: Lower-rated reviews tend to get more helpful votes.


---

## Part 2 — pandas: Loading and Manipulating a Text Dataset (30 points)

pandas is the standard tool for loading, inspecting, and wrangling text datasets before they enter an NLP pipeline. In this part you will perform a series of operations on the same review dataset, producing named variables that the autograder will check.

### 2.1 — Inspect the DataFrame

In [129]:
def inspect_dataframe(dataframe):
    """Return the (rows, columns) shape of a DataFrame.

    Args:
        dataframe (pd.DataFrame): Frame to inspect.

    Returns:
        tuple[int, int]: The frame's shape.

    Raises:
        TypeError: If `dataframe` is not a DataFrame.
    """
    require_columns(dataframe)  # no named columns required, type check only
    return dataframe.shape

df_shape = inspect_dataframe(df)

print(f"df_shape: {df_shape}")
print(f"\nColumn dtypes:")
print(df.dtypes)
# The lab brief asks for missing values as part of the first look, so it belongs
# in the same cell as shape and dtypes rather than being deferred.
print(f"\nMissing values per column:")
print(df.isna().sum())
print(f"\nFirst 3 rows:")
df.head(3)

df_shape: (20, 5)

Column dtypes:
review_id         int64
text             object
rating            int64
category         object
helpful_votes     int64
dtype: object

Missing values per column:
review_id        0
text             0
rating           0
category         0
helpful_votes    0
dtype: int64

First 3 rows:


,review_id,text,rating,category,helpful_votes
0,1,Great sound quality and comfortable fit,5,electronics,42
1,2,Stopped working after two weeks,1,electronics,88
2,3,Exactly as described fast shipping,4,books,15


### 2.2 — Add a text length column

Character length is one of the simplest text features and a useful sanity check on your data. Add a new column called `text_length` to `df` containing the character count of each review text.

**Your task:** Complete the function. It should modify `df` in place (or return a modified copy — your design choice, but be consistent).

In [130]:
def add_text_length_column(dataframe, text_col, new_col="text_length"):
    """Add a character-count column derived from a text column, in place.

    The column is added to the object that was passed in, so the caller's frame
    is updated and the frame is also returned for chaining. Under pandas 3.0
    copy-on-write this is still safe: assigning a column on a DataFrame you hold
    a direct reference to is not chained assignment, which is the pattern CoW
    turned into a no-op.

    Args:
        dataframe (pd.DataFrame): Frame to modify.
        text_col (str): Column holding the text.
        new_col (str): Name of the column to create. Defaults to "text_length".

    Returns:
        pd.DataFrame: The same frame, modified.

    Raises:
        KeyError: If `text_col` is absent.
    """
    require_columns(dataframe, text_col)
    # .str.len() is vectorised and yields NaN for missing values instead of
    # raising, which is what we want: cleaning is a separate step (Part 4.1).
    dataframe[new_col] = dataframe[text_col].str.len()
    return dataframe

add_text_length_column(df, 'text')

print(df[['text', 'text_length']].head(5))

                                            text  text_length
0        Great sound quality and comfortable fit           39
1                Stopped working after two weeks           31
2             Exactly as described fast shipping           34
3                 Poor build quality feels cheap           30
4  Absolutely love this product highly recommend           45


### 2.3 — Mean text length

Compute the mean text length across all reviews, rounded to 2 decimal places.

**Your task:** Produce `mean_text_length`.

In [131]:
def compute_mean_text_length(dataframe, length_col, ndigits=2):
    """Mean of a numeric length column, rounded.

    Args:
        dataframe (pd.DataFrame): Frame containing the column.
        length_col (str): Numeric column to average.
        ndigits (int): Decimal places to round to. Defaults to 2.

    Returns:
        float: The rounded mean.

    Raises:
        KeyError: If `length_col` is absent.
        ValueError: If the column is empty or entirely missing.
    """
    require_columns(dataframe, length_col)
    mean_value = dataframe[length_col].mean()
    if pd.isna(mean_value):
        raise ValueError(f"'{length_col}' has no non-missing values to average.")
    return round(float(mean_value), ndigits)

mean_text_length = compute_mean_text_length(df, 'text_length')

print(f"mean_text_length: {mean_text_length}")

mean_text_length: 34.55


### 2.4 — Longest review

Find the text of the review with the highest character count.

**Your task:** Produce `longest_review` (the text string, not the row).

In [132]:
def find_longest_text(dataframe, text_col, length_col):
    """Return the text of the row with the largest value in `length_col`.

    Ties go to the first occurrence, which is what idxmax() returns. Worth
    knowing rather than discovering later: on a tie this is not "the" answer,
    it is "an" answer.

    Args:
        dataframe (pd.DataFrame): Frame to search.
        text_col (str): Column holding the text to return.
        length_col (str): Numeric column to maximise.

    Returns:
        str: The text of the longest row.

    Raises:
        ValueError: If the frame is empty.
    """
    require_columns(dataframe, text_col, length_col)
    if dataframe.empty:
        raise ValueError("Cannot find the longest text in an empty DataFrame.")
    return dataframe.loc[dataframe[length_col].idxmax(), text_col]

longest_review = find_longest_text(df, 'text', 'text_length')

print(f"longest_review: '{longest_review}'")
print(f"length: {len(longest_review)} characters")

longest_review: 'Absolutely love this product highly recommend'
length: 45 characters


### 2.5 — Category counts

Count how many reviews belong to each category.

**Your task:** Produce `category_counts` as a pandas Series sorted in descending order.

In [133]:
def count_by_category(dataframe, category_col):
    """Count rows per category, largest count first.

    value_counts() already returns the counts sorted descending, so no second
    sort is applied — an extra .sort_values() here would be a no-op that implies
    the reader cannot trust the default.

    Args:
        dataframe (pd.DataFrame): Frame to count over.
        category_col (str): Column holding the category labels.

    Returns:
        pd.Series: Counts indexed by category, descending.

    Raises:
        KeyError: If `category_col` is absent.
    """
    require_columns(dataframe, category_col)
    return dataframe[category_col].value_counts()

category_counts = count_by_category(df, 'category')

print("category_counts:")
print(category_counts)

category_counts:
category
electronics    6
home           5
clothing       5
books          4
Name: count, dtype: int64


### 2.6 — Filter by text length

Return a filtered DataFrame containing only reviews whose text is longer than a given threshold.

**Your task:** Complete the function and produce `long_reviews_df`.

In [134]:
LENGTH_THRESHOLD = 35  # do not change

def filter_by_length(dataframe, length_col, threshold):
    """Return the rows whose length column is strictly greater than a threshold.

    A copy is returned, not a view, so the caller can add columns to the result
    without reaching back into the source frame.

    Args:
        dataframe (pd.DataFrame): Frame to filter.
        length_col (str): Numeric column to compare.
        threshold (int | float): Exclusive lower bound.

    Returns:
        pd.DataFrame: A new frame of the matching rows.

    Raises:
        KeyError: If `length_col` is absent.
    """
    require_columns(dataframe, length_col)
    return dataframe.loc[dataframe[length_col] > threshold].copy()

long_reviews_df = filter_by_length(df, 'text_length', LENGTH_THRESHOLD)

print(f"Reviews with text_length > {LENGTH_THRESHOLD}: {len(long_reviews_df)}")
long_reviews_df[['text', 'text_length', 'rating']].head()

Reviews with text_length > 35: 9


,text,text_length,rating
0,Great sound quality and comfortable fit,39,5
4,Absolutely love this product highly recommend,45,5
5,Instructions were confusing but product works,45,3
6,Battery life is impressive lasts all day,40,5
7,Returned immediately not as pictured,36,1


### 2.7 — Filter by rating

Return a filtered DataFrame containing only reviews with a rating at or above a given threshold.

**Your task:** Complete the function and produce `high_rated_df`.

In [135]:
RATING_FILTER = 4  # do not change

def filter_by_rating(dataframe, rating_col, min_rating):
    """Return the rows whose rating is at or above a minimum.

    Note the asymmetry with filter_by_length above: the brief specifies "at or
    above" here and "longer than" there, so this bound is inclusive and that one
    is not. Off-by-one boundary errors in filters are silent, so the comparison
    operator is worth stating out loud.

    Args:
        dataframe (pd.DataFrame): Frame to filter.
        rating_col (str): Numeric rating column.
        min_rating (int | float): Inclusive lower bound.

    Returns:
        pd.DataFrame: A new frame of the matching rows.

    Raises:
        KeyError: If `rating_col` is absent.
    """
    require_columns(dataframe, rating_col)
    return dataframe.loc[dataframe[rating_col] >= min_rating].copy()

high_rated_df = filter_by_rating(df, 'rating', RATING_FILTER)

print(f"Reviews with rating >= {RATING_FILTER}: {len(high_rated_df)}")
high_rated_df[['text', 'rating', 'category']].head()

Reviews with rating >= 4: 10


,text,rating,category
0,Great sound quality and comfortable fit,5,electronics
2,Exactly as described fast shipping,4,books
4,Absolutely love this product highly recommend,5,clothing
6,Battery life is impressive lasts all day,5,electronics
8,Good value for the price,4,books


### 2.8 — Average rating by category

Compute the mean rating for each category, rounded to 2 decimal places, sorted from highest to lowest.

**Your task:** Produce `avg_rating_by_category` as a pandas Series.

In [136]:
def avg_rating_by_group(dataframe, group_col, rating_col, ndigits=2):
    """Mean rating per group, rounded, highest first.

    Args:
        dataframe (pd.DataFrame): Frame to aggregate.
        group_col (str): Column to group by.
        rating_col (str): Numeric column to average.
        ndigits (int): Decimal places to round to. Defaults to 2.

    Returns:
        pd.Series: Mean rating indexed by group, descending.

    Raises:
        KeyError: If either column is absent.
    """
    require_columns(dataframe, group_col, rating_col)
    return (dataframe.groupby(group_col)[rating_col]
            .mean()
            .round(ndigits)
            .sort_values(ascending=False))

avg_rating_by_category = avg_rating_by_group(df, 'category', 'rating')

print("avg_rating_by_category:")
print(avg_rating_by_category)
print(f"\nGroup sizes (means over 4-6 reviews are noisy):")
print(df.groupby('category').size().sort_values(ascending=False))

avg_rating_by_category:
category
books          4.5
clothing       3.4
electronics    3.0
home           2.6
Name: rating, dtype: float64

Group sizes (means over 4-6 reviews are noisy):
category
electronics    6
clothing       5
home           5
books          4
dtype: int64


---

## Part 3 — NLTK: Frequency Distribution over a Corpus (23 points)

A frequency distribution (or FreqDist in NLTK) is one of the most fundamental tools in NLP. It counts how often each word type appears in a text, which underpins everything from basic corpus statistics to TF-IDF and language modeling.

In this part you will work with a excerpt from the Brown corpus — a standard reference corpus included with NLTK — to build a frequency distribution and extract statistics from it.

### 3.1 — Load and tokenize a corpus excerpt

In [137]:
from nltk.corpus import brown
from nltk import FreqDist
from nltk.tokenize import word_tokenize

# We use the 'news' category of the Brown corpus.
# This is a stable, well-known reference corpus.
CORPUS_CATEGORY = 'news'

def require_freq_dist(freq_dist):
    """Check that a value is an nltk.FreqDist and return it.

    Args:
        freq_dist (FreqDist): Candidate distribution.

    Returns:
        FreqDist: The same object.

    Raises:
        TypeError: If `freq_dist` is not a FreqDist.
    """
    if not isinstance(freq_dist, FreqDist):
        raise TypeError(f"Expected an nltk.FreqDist, got {type(freq_dist).__name__}.")
    return freq_dist

def load_corpus_tokens(corpus, category=None):
    """Read an NLTK corpus (optionally one category) as lowercased tokens.

    Case folding merges "The" and "the" into a single type. That is right for
    counting word forms, but it is not free: it destroys the capitalisation cue
    a named-entity tagger would rely on. Jurafsky & Martin (2026, ch. 2) make the
    same point — whether `They` and `they` are one type "depends on the task".

    Punctuation is deliberately left in. The Brown reader emits `,` and `.` as
    their own tokens, and Part 4.4 measures what removing them does rather than
    assuming it is an improvement.

    Args:
        corpus: An NLTK corpus reader (e.g. nltk.corpus.brown).
        category (str | None): Category to read. None reads the whole corpus.

    Returns:
        list[str]: Lowercased tokens in corpus order.

    Raises:
        ValueError: If the category yields no tokens.
    """
    words = corpus.words() if category is None else corpus.words(categories=category)
    tokens = [word.lower() for word in words]
    if not tokens:
        raise ValueError(f"Corpus category {category!r} produced no tokens.")
    return tokens

corpus_tokens = load_corpus_tokens(brown, CORPUS_CATEGORY)

print(f"Total tokens in '{CORPUS_CATEGORY}' category: {len(corpus_tokens)}")
print(f"First 20 tokens: {corpus_tokens[:20]}")

Total tokens in 'news' category: 100554
First 20 tokens: ['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', "atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that']


### 3.2 — Build a frequency distribution

**Your task:** Build an `nltk.FreqDist` from `corpus_tokens` and assign it to `fdist`.

In [138]:
def build_freq_dist(tokens):
    """Build an nltk.FreqDist over a sequence of tokens.

    The string guard matters. FreqDist accepts any iterable, and a bare string is
    an iterable of characters — so passing one produces a perfectly valid-looking
    distribution over letters instead of words, with no error. That is exactly
    the kind of silent failure that survives into production.

    Args:
        tokens (Sequence[str]): Tokens to count.

    Returns:
        FreqDist: Counts keyed by token.

    Raises:
        TypeError: If `tokens` is a single string.
        ValueError: If `tokens` is empty.
    """
    if isinstance(tokens, str):
        raise TypeError("Pass a sequence of tokens, not one string — FreqDist over "
                        "a string counts characters.")
    freq_dist = FreqDist(tokens)
    if freq_dist.N() == 0:
        raise ValueError("No tokens to count.")
    return freq_dist

fdist = build_freq_dist(corpus_tokens)

print(f"Vocabulary size (unique types): {len(fdist)}")
print(f"Total token count: {fdist.N()}")
print(f"\nMost common 5 words:")
for word, count in fdist.most_common(5):
    print(f"  {word!r:20s} {count:6d}")

Vocabulary size (unique types): 13112
Total token count: 100554

Most common 5 words:
  'the'                  6386
  ','                    5188
  '.'                    4030
  'of'                   2861
  'and'                  2186


### 3.3 — Vocabulary size

**Your task:** Produce `vocab_size` — the number of unique word types in the distribution.

In [139]:
def get_vocab_size(freq_dist):
    """Number of distinct types (|V|) in a frequency distribution.

    len() on a FreqDist counts keys, not occurrences — that is the vocabulary
    size. The number of running words is freq_dist.N().

    Args:
        freq_dist (FreqDist): Distribution to measure.

    Returns:
        int: The vocabulary size |V|.

    Raises:
        TypeError: If `freq_dist` is not a FreqDist.
    """
    require_freq_dist(freq_dist)
    return len(freq_dist)

vocab_size = get_vocab_size(fdist)

print(f"vocab_size: {vocab_size}")
print(f"instances N: {fdist.N()}")
print(f"type/token ratio: {vocab_size / fdist.N():.4f}")

vocab_size: 13112
instances N: 100554
type/token ratio: 0.1304


### 3.4 — Top 10 most common words

**Your task:** Produce `top_10_words` — a list of (word, count) tuples for the 10 most frequent tokens.

In [140]:
def get_top_n(freq_dist, n):
    """Return the `n` most frequent (token, count) pairs, most frequent first.

    Args:
        freq_dist (FreqDist): Distribution to read.
        n (int): How many pairs to return. Must be >= 1.

    Returns:
        list[tuple[str, int]]: Up to `n` (token, count) pairs.

    Raises:
        TypeError: If `n` is not an integer.
        ValueError: If `n` is less than 1.
    """
    require_freq_dist(freq_dist)
    if not isinstance(n, int) or isinstance(n, bool):
        raise TypeError(f"n must be an int, got {type(n).__name__}.")
    if n < 1:
        raise ValueError(f"n must be at least 1, got {n}.")
    return freq_dist.most_common(n)

top_10_words = get_top_n(fdist, 10)

print("top_10_words:")
for word, count in top_10_words:
    print(f"  {word!r:20s} {count:6d}")

top_10_words:
  'the'                  6386
  ','                    5188
  '.'                    4030
  'of'                   2861
  'and'                  2186
  'to'                   2144
  'a'                    2130
  'in'                   2020
  'for'                   969
  'that'                  829


### 3.5 — Hapax legomena

A **hapax legomenon** (plural: hapax legomena) is a word that appears exactly once in the corpus. Hapax legomena make up a large fraction of any natural language corpus and are important for understanding vocabulary coverage and language model challenges.

**Your task:** Produce `hapax_count` — the number of tokens with frequency exactly 1. Use `FreqDist.hapaxes()`.

In [141]:
def count_hapaxes(freq_dist):
    """Number of hapax legomena — types occurring exactly once.

    Args:
        freq_dist (FreqDist): Distribution to measure.

    Returns:
        int: Count of types with frequency 1.

    Raises:
        TypeError: If `freq_dist` is not a FreqDist.
    """
    require_freq_dist(freq_dist)
    return len(freq_dist.hapaxes())

hapax_count = count_hapaxes(fdist)

print(f"hapax_count: {hapax_count}")
print(f"As a fraction of vocabulary: {hapax_count / vocab_size:.1%}")
print(f"As a fraction of all tokens: {hapax_count / fdist.N():.1%}")

hapax_count: 6834
As a fraction of vocabulary: 52.1%
As a fraction of all tokens: 6.8%


### 3.6 — Frequency of a specific word

**Your task:** Complete a function that returns the frequency of any given word in the distribution. Produce `freq_the` for the word `'the'`.

In [142]:
def get_word_frequency(freq_dist, word):
    """Raw count of one word in a distribution.

    FreqDist inherits from collections.Counter, so a missing key returns 0
    rather than raising KeyError. That is convenient but it hides typos, so the
    lookup is normalised to match how the distribution was built (lowercased) and
    a miss is reported by the caller rather than passing silently as a real zero.

    Args:
        freq_dist (FreqDist): Distribution to read.
        word (str): Word to look up.

    Returns:
        int: Count of `word`, or 0 if it does not occur.

    Raises:
        TypeError: If `word` is not a string.
    """
    require_freq_dist(freq_dist)
    if not isinstance(word, str):
        raise TypeError(f"word must be a str, got {type(word).__name__}.")
    return freq_dist[word.lower()]

freq_the = get_word_frequency(fdist, 'the')

print(f"Frequency of 'the': {freq_the}")
print(f"As a fraction of all tokens: {freq_the / fdist.N():.4f}")
print(f"Sanity check on an absent word — 'transformer': "
      f"{get_word_frequency(fdist, 'transformer')} (0 means absent, not an error)")

Frequency of 'the': 6386
As a fraction of all tokens: 0.0635
Sanity check on an absent word — 'transformer': 0 (0 means absent, not an error)


---

## Part 4 — Completing the lab brief

The Canvas brief asks for three things the scaffold has no stub for:

1. clean any **missing text rows** (4.1),
2. add **at least two** derived columns (4.2),
3. report the **20** most frequent words, not 10 (4.3).

Part 5 then assembles them into the pipeline the brief's last item asks for.

### 4.1 — Missing and blank text

The provided dataset has no gaps, which makes it a bad test of a cleaning step.
So the functions are written here and then exercised on a deliberately damaged
copy, where I know what the right answer is. `df` itself is never modified.

In [143]:
def audit_missing(dataframe):
    """Count missing values per column.

    Args:
        dataframe (pd.DataFrame): Frame to audit.

    Returns:
        pd.Series: Missing-value count per column, indexed by column name.

    Raises:
        TypeError: If `dataframe` is not a DataFrame.
    """
    require_columns(dataframe)
    return dataframe.isna().sum()


def drop_missing_text(dataframe, text_col):
    """Drop rows whose text is missing or blank, returning a cleaned copy.

    Two different defects look identical downstream: a true NaN, and a string of
    nothing but whitespace. Only the first is caught by dropna(). The second
    survives, reports a text_length of 0, and quietly drags the mean down. Both
    are removed here.

    Args:
        dataframe (pd.DataFrame): Frame to clean.
        text_col (str): Column holding the text.

    Returns:
        tuple[pd.DataFrame, int]: (cleaned copy, number of rows removed).

    Raises:
        KeyError: If `text_col` is absent.
    """
    require_columns(dataframe, text_col)
    rows_before = len(dataframe)
    text = dataframe[text_col]
    is_usable = text.notna() & (text.astype("string").str.strip().str.len() > 0)
    cleaned = dataframe.loc[is_usable].copy()
    return cleaned, rows_before - len(cleaned)


# Build a damaged copy so the cleaning step has something to find.
df_with_gaps = df.copy()
df_with_gaps.loc[2, 'text'] = None      # a true missing value
df_with_gaps.loc[5, 'text'] = "   "     # blank, but not missing — dropna() misses this

print("Missing values in the damaged copy:")
print(audit_missing(df_with_gaps))

cleaned_df, rows_removed = drop_missing_text(df_with_gaps, 'text')
print(f"\ndropna() alone would remove: {len(df_with_gaps) - len(df_with_gaps.dropna(subset=['text']))} row(s)")
print(f"drop_missing_text removed:   {rows_removed} row(s)  <- catches the blank string too")
print(f"Rows remaining: {len(cleaned_df)}")

print(f"\nMissing values in the real dataset (unchanged): "
      f"{int(audit_missing(df).sum())} across all columns")

Missing values in the damaged copy:
review_id        0
text             1
rating           0
category         0
helpful_votes    0
text_length      0
dtype: int64

dropna() alone would remove: 1 row(s)
drop_missing_text removed:   2 row(s)  <- catches the blank string too
Rows remaining: 18

Missing values in the real dataset (unchanged): 0 across all columns


### 4.2 — Derived columns

`text_length` came from task 2.2. The brief asks for at least two, so `word_count`
and `avg_word_length` are added here.

These go on `reviews_df`, a copy. `df` has already been measured by the graded
tasks (`df_shape`, `long_reviews_df`, `high_rated_df`), and adding columns to it
now would change its shape after the fact. Keeping the graded frame frozen and
doing feature work on a copy costs one line and removes the whole class of
problem.

In [144]:
reviews_df = df.copy()   # graded `df` stays frozen at its post-task-2.2 shape


def add_word_count_column(dataframe, text_col, new_col="word_count"):
    """Add a whitespace-token count column.

    Splitting on whitespace is a deliberate first approximation, not a tokenizer.
    It will count "shipping," and "shipping" as two different words and it will
    split "don't" nowhere. That is fine for a length feature and wrong for a
    vocabulary — which is why Part 3 uses NLTK's corpus tokens instead.

    Args:
        dataframe (pd.DataFrame): Frame to modify.
        text_col (str): Column holding the text.
        new_col (str): Name of the column to create.

    Returns:
        pd.DataFrame: The same frame, modified.
    """
    require_columns(dataframe, text_col)
    dataframe[new_col] = dataframe[text_col].str.split().str.len()
    return dataframe


def add_avg_word_length_column(dataframe, length_col, count_col,
                               new_col="avg_word_length", ndigits=2):
    """Add mean characters-per-word, guarding against a zero word count.

    A review with no words would otherwise divide by zero. `.where(counts > 0)`
    turns the denominator into NA first, so the result is a visible NaN rather
    than an inf that propagates into every later mean.

    Args:
        dataframe (pd.DataFrame): Frame to modify.
        length_col (str): Character-count column.
        count_col (str): Word-count column.
        new_col (str): Name of the column to create.
        ndigits (int): Decimal places to round to.

    Returns:
        pd.DataFrame: The same frame, modified.
    """
    require_columns(dataframe, length_col, count_col)
    counts = dataframe[count_col]
    dataframe[new_col] = (dataframe[length_col] / counts.where(counts > 0)).round(ndigits)
    return dataframe


add_word_count_column(reviews_df, 'text')
add_avg_word_length_column(reviews_df, 'text_length', 'word_count')

print(reviews_df[['text', 'text_length', 'word_count', 'avg_word_length']].head())
print(f"\nreviews_df shape: {reviews_df.shape}   (df is untouched at {df.shape})")
print("\nFeature summary:")
print(reviews_df[['text_length', 'word_count', 'avg_word_length', 'rating']].describe().round(2))

                                            text  text_length  word_count  \
0        Great sound quality and comfortable fit           39           6   
1                Stopped working after two weeks           31           5   
2             Exactly as described fast shipping           34           5   
3                 Poor build quality feels cheap           30           5   
4  Absolutely love this product highly recommend           45           6   

   avg_word_length  
0              6.5  
1              6.2  
2              6.8  
3              6.0  
4              7.5  

reviews_df shape: (20, 8)   (df is untouched at (20, 6))

Feature summary:
       text_length  word_count  avg_word_length  rating
count        20.00        20.0            20.00   20.00
mean         34.55         5.5             6.36    3.30
std           6.02         1.0             1.05    1.49
min          24.00         4.0             4.80    1.00
25%          30.00         5.0             5.63    2.00

### 4.3 — The 20 most frequent words

The brief asks for 20. Raw counts alone are hard to read, so each row also carries
its share of all tokens and the running cumulative share — which is the number
that actually says something.

In [145]:
def top_n_table(freq_dist, n):
    """Tabulate the n most frequent types with share and cumulative share.

    Args:
        freq_dist (FreqDist): Distribution to summarise.
        n (int): Number of rows.

    Returns:
        pd.DataFrame: Columns rank, word, count, pct_of_tokens, cumulative_pct.
    """
    pairs = get_top_n(freq_dist, n)
    total = freq_dist.N()
    table = pd.DataFrame(pairs, columns=["word", "count"])
    table.insert(0, "rank", range(1, len(table) + 1))
    table["pct_of_tokens"] = (100 * table["count"] / total).round(2)
    table["cumulative_pct"] = table["pct_of_tokens"].cumsum().round(2)
    return table


top_20_words = get_top_n(fdist, 20)
top_20_table = top_n_table(fdist, 20)

print(top_20_table.to_string(index=False))
print(f"\nThose {len(top_20_table)} types are {len(top_20_table) / vocab_size:.3%} of the vocabulary "
      f"but {top_20_table['cumulative_pct'].iloc[-1]:.1f}% of all running words.")

 rank word  count  pct_of_tokens  cumulative_pct
    1  the   6386           6.35            6.35
    2    ,   5188           5.16           11.51
    3    .   4030           4.01           15.52
    4   of   2861           2.85           18.37
    5  and   2186           2.17           20.54
    6   to   2144           2.13           22.67
    7    a   2130           2.12           24.79
    8   in   2020           2.01           26.80
    9  for    969           0.96           27.76
   10 that    829           0.82           28.58
   11   is    733           0.73           29.31
   12   ``    732           0.73           30.04
   13  was    717           0.71           30.75
   14   ''    702           0.70           31.45
   15   on    691           0.69           32.14
   16   he    642           0.64           32.78
   17   at    636           0.63           33.41
   18 with    567           0.56           33.97
   19   be    526           0.52           34.49
   20   as    517   

---

## Part 5 — The assembled pipeline

The brief's last item is to "assemble the steps above into a clean, documented
pipeline that loads data, cleans it, computes basic features and outputs a
frequency distribution."

Everything it needs already exists as a function, so the pipeline is composition,
not new logic — which is the point of having written it that way. It takes the
CSV text and the corpus as arguments rather than reaching for the globals, so the
same function runs on a different dataset without being edited.

In [146]:
def run_text_pipeline(csv_text, corpus_tokens, text_col="text",
                      category_col="category", rating_col="rating", top_n=20):
    """Load, clean, feature-engineer and profile a text dataset end to end.

    Stages, in order:
      1. load   — parse CSV text into a DataFrame
      2. clean  — drop rows with missing or blank text
      3. feature— add text_length, word_count, avg_word_length
      4. count  — build a FreqDist over the supplied corpus tokens

    The corpus tokens are a parameter, not a hard-coded call to brown.words(),
    so the same pipeline profiles Reuters, Brown or a future dataset unchanged.

    Args:
        csv_text (str): Raw CSV content.
        corpus_tokens (Sequence[str]): Tokens for the frequency distribution.
        text_col (str): Name of the text column.
        category_col (str): Name of the category column.
        rating_col (str): Name of the rating column.
        top_n (int): How many frequent words to report.

    Returns:
        dict: Keys `data` (cleaned DataFrame), `rows_removed` (int),
            `freq_dist` (FreqDist) and `report` (dict of summary numbers).

    Raises:
        TypeError: If `csv_text` is not a string.
        ValueError: If cleaning removes every row.
    """
    if not isinstance(csv_text, str):
        raise TypeError(f"csv_text must be a str, got {type(csv_text).__name__}.")

    # 1. load
    data = pd.read_csv(io.StringIO(csv_text))
    require_columns(data, text_col, category_col, rating_col)

    # 2. clean
    data, rows_removed = drop_missing_text(data, text_col)
    if data.empty:
        raise ValueError("Cleaning removed every row; nothing left to profile.")

    # 3. feature
    add_text_length_column(data, text_col)
    add_word_count_column(data, text_col)
    add_avg_word_length_column(data, "text_length", "word_count")

    # 4. count
    freq_dist = build_freq_dist(corpus_tokens)

    report = {
        "rows_in": len(data) + rows_removed,
        "rows_removed": rows_removed,
        "rows_out": len(data),
        "mean_text_length": compute_mean_text_length(data, "text_length"),
        "mean_word_count": compute_mean_text_length(data, "word_count"),
        "categories": count_by_category(data, category_col).to_dict(),
        "mean_rating_by_category": avg_rating_by_group(
            data, category_col, rating_col).to_dict(),
        "corpus_instances": freq_dist.N(),
        "corpus_types": len(freq_dist),
        "corpus_hapaxes": count_hapaxes(freq_dist),
        f"top_{top_n}": get_top_n(freq_dist, top_n),
    }
    return {"data": data, "rows_removed": rows_removed,
            "freq_dist": freq_dist, "report": report}


pipeline_result = run_text_pipeline(REVIEWS_CSV, corpus_tokens)
pipeline_report = pipeline_result["report"]

print("PIPELINE REPORT")
print("=" * 62)
for key, value in pipeline_report.items():
    if key.startswith("top_"):
        preview = ", ".join(f"{word}({count})" for word, count in value[:8])
        print(f"{key:>24}: {preview}, ...")
    else:
        print(f"{key:>24}: {value}")
print("=" * 62)
print(f"\nCleaned data, first 3 rows:")
print(pipeline_result["data"].head(3).to_string(index=False))

PIPELINE REPORT
                 rows_in: 20
            rows_removed: 0
                rows_out: 20
        mean_text_length: 34.55
         mean_word_count: 5.5
              categories: {'electronics': 6, 'home': 5, 'clothing': 5, 'books': 4}
 mean_rating_by_category: {'books': 4.5, 'clothing': 3.4, 'electronics': 3.0, 'home': 2.6}
        corpus_instances: 100554
            corpus_types: 13112
          corpus_hapaxes: 6834
                  top_20: the(6386), ,(5188), .(4030), of(2861), and(2186), to(2144), a(2130), in(2020), ...

Cleaned data, first 3 rows:
 review_id                                    text  rating    category  helpful_votes  text_length  word_count  avg_word_length
         1 Great sound quality and comfortable fit       5 electronics             42           39           6              6.5
         2         Stopped working after two weeks       1 electronics             88           31           5              6.2
         3      Exactly as described fast sh

In [147]:
# The pipeline must survive bad input, not just the happy path.
# Three cases the graded scaffold never exercises:

DAMAGED_CSV = REVIEWS_CSV.replace(
    "3,Exactly as described fast shipping,4,books,15",
    "3,,4,books,15"                       # empty text field
).replace(
    "9,Good value for the price,4,books,18",
    "9,   ,4,books,18"                    # whitespace-only text field
)

damaged_result = run_text_pipeline(DAMAGED_CSV, corpus_tokens)
print(f"Damaged CSV: {damaged_result['report']['rows_in']} rows in, "
      f"{damaged_result['report']['rows_removed']} removed, "
      f"{damaged_result['report']['rows_out']} out.")

for label, bad_input in [
    ("csv_text as a list", (["not", "a", "csv"], corpus_tokens)),
    ("corpus as one string", (REVIEWS_CSV, "a single string of text")),
]:
    try:
        run_text_pipeline(*bad_input)
    except (TypeError, ValueError) as error:
        print(f"{label:<22} -> {type(error).__name__}: {error}")

Damaged CSV: 20 rows in, 2 removed, 18 out.
csv_text as a list     -> TypeError: csv_text must be a str, got list.
corpus as one string   -> TypeError: Pass a sequence of tokens, not one string — FreqDist over a string counts characters.


---

## References

Jurafsky, D., & Martin, J. H. (2026). *Speech and language processing: An introduction to natural language processing, computational linguistics, and speech recognition with language models* (3rd ed., draft of August 19, 2026), Chapter 2: Words and Tokens. https://web.stanford.edu/~jurafsky/slp3/

Kučera, H., & Francis, W. N. (1967). *Computational analysis of present-day American English*. Brown University Press.

The pandas development team. (2026). *pandas 3.0.0 release notes*. https://pandas.pydata.org/docs/whatsnew/v3.0.0.html


---

## Submission checklist

Before uploading to Canvas:

- [x] All cells run top-to-bottom without errors (use **Runtime → Run all**)
- [x] All named variables are defined: `ratings_array`, `helpful_array`, `total_ratings`, `mean_rating`, `normalized_ratings`, `above_threshold_mask`, `filtered_ratings`, `ratings_helpful_corr`, `df_shape`, `mean_text_length`, `longest_review`, `category_counts`, `long_reviews_df`, `high_rated_df`, `avg_rating_by_category`, `corpus_tokens`, `fdist`, `vocab_size`, `top_10_words`, `hapax_count`, `freq_the`
- [x] Every function has a complete docstring
- [x] No hard-coded answer values (your functions must compute the answers, not return literals)
- [x] File is saved as `Lab_1.ipynb`